In [ ]:
from astropy.io import fits
from astropy.table import Table as t
import numpy as np
import seaborn as sns
from umap import UMAP as u

import h5py
from astropy.convolution import Gaussian1DKernel, convolve
from collections import defaultdict
import os
from sklearn.datasets import make_blobs


from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import networkx as nx
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
plt.rcParams['text.usetex'] = True
plt.style.use('./data/plots/desi.mplstyle')
plt.rcParams['figure.dpi'] = 360

In [ ]:
X, _ = make_blobs(n_samples=600, centers=3, cluster_std=4.)

reducer = u(
    n_neighbors=100,
    min_dist=0.05,
    spread=0.2
)
X_emb = reducer.fit_transform(X)

radius = 0.1
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)

n_components, labels = connected_components(adj, directed=False)

rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

pos = {i: X_emb[i] for i in range(len(X_emb))}
cmap = ListedColormap(sns.color_palette("mako", n_components))

plt.figure()
nx.draw_networkx_edges(G, pos, alpha=0.6, width=0.5)

for cluster_id in range(n_components):
    nodelist = np.where(labels == cluster_id)[0]
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=nodelist,
        node_size=60,
        node_color=[cmap(cluster_id)],
        label=f'Cluster {cluster_id+1}',
        alpha=0.8
    )

plt.axis('off')
# plt.legend(markerscale=1, fontsize='small', loc='upper right', title='Clusters')
# plt.title('UMAP + FoF como Grafo con colores únicos por cluster')
plt.tight_layout()
plt.show()


In [ ]:
import os
import sys
import subprocess

import numpy as np

from astropy.io import fits
from astropy.convolution import convolve, Gaussian1DKernel

from desimodel.footprint import radec2pix      # For getting healpix values #! install one by one
import desispec.io                             # Input/Output functions related to DESI spectra
from desispec import coaddition                # Functions related to coadding the spectra

# DESI targeting masks -
from desitarget.sv1 import sv1_targetmask    # For SV1
from desitarget.sv2 import sv2_targetmask    # For SV2
from desitarget.sv3 import sv3_targetmask    # For SV3
from desitarget.targets import desi_mask

from astropy.io import fits
from astropy.table import Table as t

# plt.style.use('./plots/desi.mplstyle')

In [ ]:
data_path = '../../desi_data/10256/20211110/coadd-0-10256-thru20211110.fits'

In [ ]:
file = fits.open(data_path)
file[1].columns

In [ ]:
FIBER, 